# XR Client Utilities

Examples of defining user commands that can be invoked by the XR client's command menu.

## Setup runner

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../systems/openmm/17-ala.openmm.zip")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="xr client utilities example")
imd_runner.load(0)

## Runner utilities
You can wrap a set of utilities around the runner to add extra commands:

In [2]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

## XR client notification
The XR client defines a remote command to send it notifications (displayed over the controller); the utilities provide a convenience method `notify_all` to call all defined commands of this type.

We define an additional local notification command:

In [3]:
def test_notify(message):
    print(message)


imd_runner.app_server.register_command("test/notify", test_notify, icon="📢")

The `notify_all` triggers the notification command of any connected XR clients, in addition to the command we just defined:

In [4]:
utilities.notify_all("TESTING!")

TESTING!


## Interaction modes

The utilities provide a basic system of interaction "modes" that define user commands for switching between different custom behaviors in response to iMD interactions:

In [5]:
utilities.modes.add_normal_mode()

Defining a mode where every user interaction triggers a notification in the XR client with information about the interacted atom from MDAnalysis:

<video controls src="./assets/info_mode.webm">

In [6]:
from nanover.imd import ParticleInteraction
from nanover.jupyter import Mode
from nanover.mdanalysis import frame_data_to_mdanalysis

# prepare info from simulation
universe = frame_data_to_mdanalysis(utilities.current_frame)


class InfoMode(Mode):
    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        atom = universe.atoms[interaction.particles[0]]
        utilities.notify_all(str(atom))

    def on_interaction_stopped(self, *, key: str, interaction: ParticleInteraction):
        pass


utilities.modes.add_mode(InfoMode(), "info", icon="ℹ️")

Defining a mode where every two user interactions trigger a notification in the XR client with the distance between the two atoms:

<video controls src="./assets/length_mode.webm">

In [7]:
import numpy as np
from nanover.imd import ParticleInteraction
from nanover.jupyter import Mode


class LengthMode(Mode):
    prev_atom = None

    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        next_atom = interaction.particles[0]

        if self.prev_atom is None:
            self.prev_atom = next_atom
            utilities.notify_all(f"{self.prev_atom} <-> [pick atom] = ???nm")
        else:
            positions = utilities.current_frame.particle_positions
            distance = np.linalg.norm(positions[self.prev_atom] - positions[next_atom])
            utilities.notify_all(f"{self.prev_atom} <-> {next_atom} = {distance:.3g}nm")
            self.prev_atom = None

    def on_interaction_stopped(self, *, key: str, interaction: ParticleInteraction):
        pass


utilities.modes.add_mode(LengthMode(), "length", icon="📏")